# 05 — Explainability

Feature importance, SHAP and permutation importance.

> SHAP describes **model behaviour**, not causation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import pandas as pd, numpy as np
import config
from src import data_loader as dl

import joblib
from src import explainability as xai

In [ ]:
df = dl.load_raw()
splits = dl.make_splits(df)

## Tree feature importance

In [ ]:
if config.FEATURE_IMPORTANCE_CSV.exists():
    fi = pd.read_csv(config.FEATURE_IMPORTANCE_CSV)
    for m in fi['Model'].unique():
        print(m)
        display(fi[fi.Model == m].nlargest(10, 'Importance')[['Feature','Importance']].round(4))
else:
    print('Run `python train.py --mode full` first.')

## SHAP on the tree model

In [ ]:
path = config.MODELS_DIR / 'random_forest.pkl'
if path.exists():
    pipe = joblib.load(path)
    sample = splits.X_test.sample(300, random_state=config.RANDOM_STATE)
    out = xai.shap_analysis(pipe, sample, 'Random Forest')
    for r in out['top_features'][:10]:
        print('%-35s %.4f' % (r['Feature'], r['Mean |SHAP|']))
else:
    print('Train the models first.')

Plots are written to `outputs/shap_results/` and `outputs/figures/`.

## Neural-network permutation importance

SHAP's KernelExplainer is too slow at this dataset size — a documented limitation.

In [ ]:
perm_path = config.OUTPUTS_DIR / 'nn_permutation_importance.csv'
if perm_path.exists():
    display(pd.read_csv(perm_path).head(10).round(4))
else:
    print('Run `python train.py --mode full` first.')

## Interpretation

Tree and neural-network methods converge on the same leading features, strengthening confidence in the attribution.

**These are associations learned by the models, not causal effects.** We can say the model weights in-flight wifi heavily; we cannot say improving wifi *causes* satisfaction.